# 01 - 探索性資料分析 (EDA)

本 notebook 對四個語音情緒資料集（RAVDESS, CREMA-D, TESS, SAVEE）進行探索性分析，
包含情緒分布、音訊時長、說話者樣本數等視覺化。

In [ ]:
import pandas as pd
import soundfile as sf
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from tqdm import tqdm

# === 專案路徑設定 ===
PROJECT_ROOT = Path("..").resolve()
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# === 載入 metadata ===
df = pd.read_csv(PROJECT_ROOT / "data" / "metadata.csv")
print(f"總樣本數：{len(df):,}")
print(f"資料集：{df['dataset'].unique().tolist()}")
print(f"情緒類別：{sorted(df['emotion'].unique().tolist())}")
df.head()

## 1. 各資料集情緒分布（分組長條圖）

In [ ]:
# 統計各資料集 × 情緒的樣本數
emotion_order = sorted(df["emotion"].unique())
dataset_order = ["RAVDESS", "CREMA-D", "TESS", "SAVEE"]

# 計算分組統計
grouped = df.groupby(["emotion", "dataset"]).size().reset_index(name="count")

fig1 = px.bar(
    grouped,
    x="emotion",
    y="count",
    color="dataset",
    barmode="group",
    category_orders={"emotion": emotion_order, "dataset": dataset_order},
    title="各資料集情緒分布",
    labels={"emotion": "情緒類別", "count": "樣本數", "dataset": "資料集"},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig1.update_layout(
    xaxis_tickangle=-30,
    legend_title_text="資料集",
    template="plotly_white",
)
fig1.write_html(FIGURES_DIR / "01_emotion_by_dataset.html")
fig1.write_image(FIGURES_DIR / "01_emotion_by_dataset.png", width=900, height=500, scale=2)
fig1.show()

## 2. 合併後整體情緒分布

In [ ]:
# 合併後的整體情緒分布
overall = df["emotion"].value_counts().sort_index().reset_index()
overall.columns = ["emotion", "count"]

fig2 = px.bar(
    overall,
    x="emotion",
    y="count",
    title="合併後整體情緒分布（四個資料集）",
    labels={"emotion": "情緒類別", "count": "樣本數"},
    color="emotion",
    color_discrete_sequence=px.colors.qualitative.Pastel,
    category_orders={"emotion": emotion_order},
    text="count",
)
fig2.update_traces(textposition="outside")
fig2.update_layout(
    showlegend=False,
    template="plotly_white",
    xaxis_tickangle=-30,
)
fig2.write_html(FIGURES_DIR / "02_overall_emotion_distribution.html")
fig2.write_image(FIGURES_DIR / "02_overall_emotion_distribution.png", width=900, height=500, scale=2)
fig2.show()

## 3. 各資料集音訊時長分布\n\n使用 `soundfile.info()` 讀取音檔 header（不載入音訊資料），快速取得時長。

In [ ]:
# 計算每筆音檔的時長（秒）
durations = []
for filepath in tqdm(df["filepath"], desc="讀取音檔時長"):
    full_path = PROJECT_ROOT / filepath
    try:
        info = sf.info(str(full_path))
        durations.append(info.duration)
    except Exception:
        durations.append(None)

df["duration"] = durations
print(f"成功讀取：{df['duration'].notna().sum():,} / {len(df):,}")
print(f"\n各資料集時長統計（秒）：")
print(df.groupby("dataset")["duration"].describe().round(2).to_string())

In [ ]:
# 各資料集音訊時長分布直方圖
fig3 = px.histogram(
    df.dropna(subset=["duration"]),
    x="duration",
    color="dataset",
    barmode="overlay",
    nbins=80,
    opacity=0.65,
    category_orders={"dataset": dataset_order},
    title="各資料集音訊時長分布",
    labels={"duration": "時長（秒）", "count": "樣本數", "dataset": "資料集"},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig3.update_layout(
    template="plotly_white",
    legend_title_text="資料集",
    xaxis_range=[0, 10],  # 大多數音檔在 10 秒以內
)
fig3.write_html(FIGURES_DIR / "03_duration_distribution.html")
fig3.write_image(FIGURES_DIR / "03_duration_distribution.png", width=900, height=500, scale=2)
fig3.show()

## 4. 各說話者樣本數分布

In [ ]:
# 計算每位說話者的樣本數
speaker_counts = (
    df.groupby(["dataset", "speaker_id"])
    .size()
    .reset_index(name="count")
)

fig4 = px.box(
    speaker_counts,
    x="dataset",
    y="count",
    color="dataset",
    points="all",  # 顯示所有資料點
    category_orders={"dataset": dataset_order},
    title="各資料集說話者樣本數分布",
    labels={"dataset": "資料集", "count": "每位說話者的樣本數"},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig4.update_layout(
    showlegend=False,
    template="plotly_white",
)
fig4.write_html(FIGURES_DIR / "04_speaker_sample_distribution.html")
fig4.write_image(FIGURES_DIR / "04_speaker_sample_distribution.png", width=900, height=500, scale=2)
fig4.show()

# 印出各資料集的說話者數與每人樣本數統計
print("各資料集說話者統計：")
for ds in dataset_order:
    subset = speaker_counts[speaker_counts["dataset"] == ds]
    print(f"  {ds:10s}: {len(subset):>3} speakers, "
          f"每人 {subset['count'].min()}-{subset['count'].max()} 筆 "
          f"(平均 {subset['count'].mean():.0f})")

## 5. 情緒 × 資料集 交叉表熱力圖

In [ ]:
# 情緒 × 資料集 交叉表
cross_tab = pd.crosstab(df["emotion"], df["dataset"])
# 重新排序欄位
cross_tab = cross_tab.reindex(columns=dataset_order, index=emotion_order)

fig5 = px.imshow(
    cross_tab,
    text_auto=True,
    color_continuous_scale="Blues",
    title="情緒 × 資料集 交叉分析熱力圖",
    labels={"x": "資料集", "y": "情緒類別", "color": "樣本數"},
    aspect="auto",
)
fig5.update_layout(
    template="plotly_white",
)
fig5.write_html(FIGURES_DIR / "05_emotion_dataset_heatmap.html")
fig5.write_image(FIGURES_DIR / "05_emotion_dataset_heatmap.png", width=700, height=500, scale=2)
fig5.show()

# 標記只在部分資料集出現的情緒
print("\n情緒覆蓋情況：")
for emotion in emotion_order:
    present_in = [ds for ds in dataset_order if cross_tab.loc[emotion, ds] > 0]
    missing_from = [ds for ds in dataset_order if cross_tab.loc[emotion, ds] == 0]
    if missing_from:
        print(f"  {emotion:10s}: 缺少於 {', '.join(missing_from)}")
    else:
        print(f"  {emotion:10s}: 四個資料集皆有")

## 6. 儲存含時長的 metadata\n\n將計算好的 `duration` 欄位一併存回 metadata，供後續 Phase 使用。

In [ ]:
# 將含時長的 metadata 存回 CSV
df.to_csv(PROJECT_ROOT / "data" / "metadata.csv", index=False)
print(f"已更新 metadata.csv，新增 duration 欄位（{df['duration'].notna().sum():,} 筆有效）")